# QuantumJam 2026 · Curva de energía de $\mathrm{H_4}$ con VQE

Cadena lineal de 4 H, espaciado uniforme $d$, base STO-3G. Enunciado en `docs/`.

## 0. Instalación

In [ ]:
%pip install -q qiskit==2.5.2 qiskit-aer==0.17.2 pyscf==2.14.0 numpy scipy pandas matplotlib

## 0.1 Contrato compartido

Nombres, firmas y convenciones que usa todo el notebook. Cada sección reemplaza su `NotImplementedError`; cambios a este contrato se avisan en el issue.

In [ ]:
# Contrato compartido: nombres, firmas y convenciones que usa todo el notebook.
# Cada sección implementa su parte; nadie cambia esto sin avisar en el issue correspondiente.
import numpy as np
import pandas as pd
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import CouplingMap

# --- Grilla y geometría -------------------------------------------------------
BASIS = "sto-3g"
D = np.round(np.concatenate([np.arange(0.6, 1.75, 0.1), np.arange(1.8, 3.01, 0.2)]), 2)  # Å, 19 puntos
STRETCHED = 1.8    # Å: desde acá hay que ganarle a CCSD
DISSOCIATED = 3.0  # Å: D_e = E(3,0) - E(R_e)
CHEM_ACC = 1.6e-3  # Ha


def h4_geometry(d: float) -> str:
    """Cadena lineal de 4 H sobre el eje z, separados por d Å (formato PySCF)."""
    return "; ".join(f"H 0 0 {i * d:.6f}" for i in range(4))


# --- Convención de qubits (Jordan-Wigner) --------------------------------------
# Orbitales espaciales MO de RHF, p = 0..3 en orden de energía.
# qubit p     = espín-orbital (p, α)
# qubit 4 + p = espín-orbital (p, β)
# |1> = ocupado. Qiskit es little-endian: el label de un estado se lee q7 ... q0.
N_ORB, N_QUBITS, N_ALPHA, N_BETA = 4, 8, 2, 2
HF_BITSTRING = "00110011"  # q0, q1 (α 0,1) y q4, q5 (β 0,1) ocupados


def alpha_qubit(p: int) -> int:
    return p


def beta_qubit(p: int) -> int:
    return N_ORB + p


# --- Transpilación y conteo de cz (igual para todos) ---------------------------
BASIS_GATES = ["cz", "sx", "x", "rz"]


def transpile_line(qc: QuantumCircuit) -> QuantumCircuit:
    return transpile(qc, basis_gates=BASIS_GATES, coupling_map=CouplingMap.from_line(qc.num_qubits),
                     optimization_level=3, seed_transpiler=0)


def cz_count(qc: QuantumCircuit) -> int:
    return transpile_line(qc).count_ops().get("cz", 0)


# --- Firmas que implementa cada sección ----------------------------------------
# Tablas compartidas (energías totales en Ha, con repulsión nuclear):
#   refs   : DataFrame  d, e_hf, e_ccsd, e_fci           (#1)
#   params : dict       d -> np.ndarray θ*               (#5)
#   vqe    : DataFrame  d, energy, sigma, shots, cz      (#6)

def references(d: float) -> dict:
    """#1. {'d', 'e_hf', 'e_ccsd', 'e_fci'}."""
    raise NotImplementedError


def qubit_hamiltonian(d: float) -> tuple[SparsePauliOp, float]:
    """#2. (H electrónico como SparsePauliOp de 8 qubits en la convención de arriba, e_nuc)."""
    raise NotImplementedError


def build_ansatz() -> QuantumCircuit:
    """#4. Circuito parametrizado de 8 qubits que arranca en HF_BITSTRING."""
    raise NotImplementedError


## 1. Referencias clásicas (HF, CCSD, FCI)

Energías de referencia en toda la grilla `D` con PySCF. **FCI** es la respuesta exacta en la base STO-3G, contra la que se mide todo. **CCSD** es el método clásico a superar en la zona estirada. Ninguna de las dos es un resultado del VQE: solo sirven para comparar y para elegir puntos de partida.

Esta celda define `references(d)` y la tabla `refs` (`d, e_hf, e_ccsd, e_fci`), que también se guarda en `results/classical.csv`. Usa el mismo procedimiento de RHF que `h4_scf` de la sección 2 (`conv_tol = 1e-11`). Las energías no dependen del signo de los orbitales. Para cualquier cosa que dependa de los orbitales (vector CI, amplitudes), usar `h4_scf(d)`.

**Decisiones**

- Se usa **RHF**, no UHF. Desde ~1,2 Å RHF tiene una inestabilidad RHF → UHF: la solución UHF baja la energía pero rompe la simetría de espín. Queremos un singlete con orbitales restringidos, que es lo que usa el Hamiltoniano de qubits. El chequeo de **estabilidad interna** (RHF → RHF) pasa en todos los puntos: no hay una solución RHF más baja que la encontrada.
- Tolerancias ajustadas (`1e-11` en SCF, `1e-10` en CCSD, `1e-12` en FCI). El código verifica la convergencia de RHF y CCSD en cada punto y que el estado FCI sea singlete (⟨S²⟩ = 0). Todos convergen con el `init_guess` por defecto, así que no hizo falta arrancar desde la densidad del punto vecino.

**Qué muestran los números**

- Cerca del equilibrio (0,6–1,1 Å) CCSD está a menos de 0,01 mHa de FCI.
- Desde ~1,1 Å CCSD **deja de ser variacional** y queda por debajo de FCI. El error es −9,1 mHa a 1,8 Å (el punto más exigente de la zona juzgada), llega a −30,6 mHa a 2,6 Å y vuelve a −23,0 mHa a 3,0 Å.
- Para ganarle a CCSD hace falta `|E_VQE − E_FCI| + σ < 9,1 mHa` a 1,8 Å y < 18 mHa desde 2,0 Å.

In [ ]:
# 1. Referencias clásicas: HF, CCSD y FCI en toda la grilla D (#1).
# Solo sirven para comparar y elegir puntos de partida; nunca son resultados.
# Las energías no dependen del signo de los orbitales: los orbitales "oficiales" (signo fijo,
# los del Hamiltoniano) son los de h4_scf(d) en la sección 2. Acá se usa el mismo procedimiento.
import os
import matplotlib.pyplot as plt
from pyscf import cc, fci, gto, scf


def references(d: float) -> dict:
    """#1. {'d', 'e_hf', 'e_ccsd', 'e_fci'} en Ha (energías totales)."""
    mol = gto.M(atom=h4_geometry(d), basis=BASIS, unit="Angstrom", charge=0, spin=0, verbose=0)
    mf = scf.RHF(mol)
    mf.conv_tol = 1e-11
    mf.kernel()
    # Al estirar, el SCF podría caer en un punto silla. Si la solución tiene una inestabilidad
    # interna (RHF -> RHF), seguimos la dirección inestable y reconvergemos (en esta grilla no pasa).
    for _ in range(5):
        mo, _, stable, _ = mf.stability(return_status=True)
        if stable:
            break
        mf.kernel(dm0=mf.make_rdm1(mo, mf.mo_occ))
    assert mf.converged, f"RHF no convergió en d={d}"

    ccsd = cc.CCSD(mf)
    ccsd.conv_tol, ccsd.max_cycle = 1e-10, 500
    ccsd.kernel()
    assert ccsd.converged, f"CCSD no convergió en d={d}"

    solver = fci.FCI(mf)
    solver.conv_tol = 1e-12
    e_fci, ci = solver.kernel()
    s2, _ = solver.spin_square(ci, N_ORB, (N_ALPHA, N_BETA))
    assert abs(s2) < 1e-6, f"FCI no es singlete en d={d}: <S²>={s2:.3g}"
    return {"d": float(d), "e_hf": mf.e_tot, "e_ccsd": ccsd.e_tot, "e_fci": e_fci}


refs = pd.DataFrame([references(d) for d in D])

# Chequeos de cordura: FCI es exacto en la base (variacional), CCSD es casi exacto cerca del equilibrio.
assert (refs.e_fci <= refs.e_hf + 1e-10).all()
near_eq = refs[(refs.d >= 0.8) & (refs.d <= 1.0)]
assert ((near_eq.e_ccsd - near_eq.e_fci).abs() < CHEM_ACC).all()

os.makedirs("results", exist_ok=True)
refs.to_csv("results/classical.csv", index=False)

err_ccsd = 1e3 * (refs.e_ccsd - refs.e_fci)  # mHa, con signo
print(refs.assign(ccsd_menos_fci_mHa=err_ccsd).to_string(
    index=False, formatters={"d": "{:.2f}".format, "ccsd_menos_fci_mHa": "{:+.3f}".format}, float_format="%.6f"))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))
ax1.plot(refs.d, refs.e_hf, "s-", ms=4, label="RHF")
ax1.plot(refs.d, refs.e_ccsd, "^-", ms=4, label="CCSD")
ax1.plot(refs.d, refs.e_fci, "o-", ms=4, color="k", label="FCI (exacta)")
ax1.axvspan(STRETCHED, D.max() + 0.05, color="0.92", zorder=0, label=f"zona estirada (d ≥ {STRETCHED} Å)")
ax1.set(xlabel="d (Å)", ylabel="E (Ha)", title="Curva de energía de H₄ (STO-3G)")
ax1.legend()

ax2.semilogy(refs.d, err_ccsd.abs(), "^-", ms=4, color="C1", label="|E_CCSD − E_FCI|")
ax2.axhline(1e3 * CHEM_ACC, ls="--", color="C2", label="precisión química (1,6 mHa)")
ax2.axvspan(STRETCHED, D.max() + 0.05, color="0.92", zorder=0)
ax2.set(xlabel="d (Å)", ylabel="error (mHa)", title="Error de CCSD: lo que hay que superar")
ax2.legend()
plt.tight_layout()
plt.show()

## 2. Hamiltoniano de qubits

In [ ]:
import copy
from functools import lru_cache

from pyscf import ao2mo, fci, gto, scf
from pyscf.fci import cistring


def _so(p: int, spin: int) -> int:
    return alpha_qubit(p) if spin == 0 else beta_qubit(p)


@lru_cache(maxsize=None)
def _jw_ladder(n_qubits: int) -> tuple[tuple[SparsePauliOp, ...], tuple[SparsePauliOp, ...]]:
    annihilation = []
    for j in range(n_qubits):
        z_string = list(range(j))
        annihilation.append(SparsePauliOp.from_sparse_list(
            [("Z" * j + "X", z_string + [j], 0.5), ("Z" * j + "Y", z_string + [j], 0.5j)],
            num_qubits=n_qubits))
    creation = [a.adjoint() for a in annihilation]
    return tuple(creation), tuple(annihilation)


def jordan_wigner(h1_so: np.ndarray, h2_so: np.ndarray | None = None, atol: float = 1e-12,
                  hermitian: bool = True) -> SparsePauliOp:
    n = h1_so.shape[0]
    cdag, a = _jw_ladder(n)
    terms = [SparsePauliOp("I" * n, 0.0)]
    for p, q in np.argwhere(np.abs(h1_so) > atol):
        terms.append(h1_so[p, q] * cdag[p] @ a[q])
    if h2_so is not None:
        for p, q, r, s in np.argwhere(np.abs(h2_so) > atol):
            if p == r or q == s:
                continue
            terms.append(0.5 * h2_so[p, q, r, s] * cdag[p] @ cdag[r] @ a[s] @ a[q])
    op = SparsePauliOp.sum(terms).simplify(atol=atol)
    if not hermitian:
        return op
    if np.max(np.abs(op.coeffs.imag), initial=0.0) > 1e-10:
        raise ValueError("Operador no hermítico (coeficientes complejos): revisar índices o usar hermitian=False")
    return SparsePauliOp(op.paulis, op.coeffs.real)


def spatial_to_spin_orbitals(h1: np.ndarray, eri: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    n_orb = h1.shape[0]
    n_so = 2 * n_orb
    h1_so = np.zeros((n_so, n_so))
    h2_so = np.zeros((n_so, n_so, n_so, n_so))
    for s1 in (0, 1):
        i = [_so(p, s1) for p in range(n_orb)]
        h1_so[np.ix_(i, i)] = h1
        for s2 in (0, 1):
            k = [_so(p, s2) for p in range(n_orb)]
            h2_so[np.ix_(i, i, k, k)] = eri
    return h1_so, h2_so


def _distance_key(d: float) -> float:
    return round(float(d), 6)


@lru_cache(maxsize=None)
def _h4_scf_cached(d: float) -> scf.hf.RHF:
    mol = gto.M(atom=h4_geometry(d), basis=BASIS, unit="Angstrom", charge=0, spin=0, verbose=0)
    mf = scf.RHF(mol)
    mf.conv_tol = 1e-11
    mf.kernel()
    if not mf.converged:
        raise RuntimeError(f"RHF no convergió en d = {d}")
    c = mf.mo_coeff
    significant = np.abs(c) > 1e-4
    if not significant.any(axis=0).all():
        raise RuntimeError(f"Orbital sin coeficientes > 1e-4 en d = {d}: no se puede fijar el signo")
    first = np.argmax(significant, axis=0)
    mf.mo_coeff = c * np.sign(c[first, np.arange(c.shape[1])])
    return mf


def h4_scf(d: float) -> scf.hf.RHF:
    mf = copy.copy(_h4_scf_cached(_distance_key(d)))
    for attr in ("mo_coeff", "mo_energy", "mo_occ"):
        setattr(mf, attr, getattr(mf, attr).copy())
    return mf


@lru_cache(maxsize=None)
def _qubit_hamiltonian_cached(d: float) -> tuple[SparsePauliOp, float]:
    mf = _h4_scf_cached(d)
    mol, c = mf.mol, mf.mo_coeff
    h1 = c.T @ mf.get_hcore() @ c
    eri = ao2mo.restore(1, ao2mo.full(mol, c), c.shape[1])
    h1_so, h2_so = spatial_to_spin_orbitals(h1, eri)
    return jordan_wigner(h1_so, h2_so), mol.energy_nuc()


def qubit_hamiltonian(d: float) -> tuple[SparsePauliOp, float]:
    op, e_nuc = _qubit_hamiltonian_cached(_distance_key(d))
    return op.copy(), e_nuc


_STRINGS = cistring.make_strings(range(N_ORB), N_ALPHA)
SECTOR = np.array([int(a) | (int(b) << N_ORB) for a in _STRINGS for b in _STRINGS])


def sector_ground_energy(op: SparsePauliOp) -> float:
    m = op.to_matrix()
    return np.linalg.eigvalsh(m[np.ix_(SECTOR, SECTOR)])[0].real


def hf_energy_from_op(op: SparsePauliOp) -> float:
    i = int(HF_BITSTRING, 2)
    return op.to_matrix()[i, i].real


def validate_hamiltonian(d: float) -> dict:
    op, e_nuc = qubit_hamiltonian(d)
    mf = h4_scf(d)
    e_fci = fci.FCI(mf).kernel()[0]
    e_hf_op = hf_energy_from_op(op) + e_nuc
    e_gs_op = sector_ground_energy(op) + e_nuc
    return {"d": float(d), "n_paulis": len(op), "e_hf": mf.e_tot, "e_fci": e_fci,
            "err_hf": abs(e_hf_op - mf.e_tot), "err_fci": abs(e_gs_op - e_fci)}


ham_check = pd.DataFrame([validate_hamiltonian(d) for d in D])
assert (ham_check[["err_hf", "err_fci"]] < 1e-8).all().all(), ham_check
ham_check


### Decisiones (#2)

**Firma.** `qubit_hamiltonian(d) -> (op, e_nuc)`, como dice el contrato 0.1 ([ADR-0001](https://github.com/Santiago-Garrote/QJ/blob/main/docs/adr/0001-firma-qubit-hamiltonian.md)). `op` es solo la parte electrónica: energía total = $\langle op \rangle + e_\text{nuc}$. El estado HF es la constante `HF_BITSTRING = "00110011"` porque no depende de $d$.

**Orden de qubits.** Qubit $p$ = espín-orbital $(p, \alpha)$, qubit $4+p$ = $(p, \beta)$, con $p = 0..3$ los orbitales RHF ordenados por energía. `HF_BITSTRING` deja ocupados q0, q1, q4, q5 (Qiskit es little-endian: el label se lee q7 … q0).

**Hamiltoniano.** Integrales en base MO de RHF: $h_{pq} = C^T h^\text{core} C$ y $(pq|rs)$ con `ao2mo` (notación química). En espín-orbitales:

$$\hat H = \sum_{PQ} h_{PQ}\, a_P^\dagger a_Q + \tfrac12 \sum_{PQRS} (PQ|RS)\, a_P^\dagger a_R^\dagger a_S a_Q,$$

con el espín conservado dentro de cada par $(P,Q)$ y $(R,S)$. Jordan-Wigner a mano: $a_j = Z_0 \cdots Z_{j-1}\,(X_j + iY_j)/2$ ([ADR-0002](https://github.com/Santiago-Garrote/QJ/blob/main/docs/adr/0002-jordan-wigner-a-mano.md)). `jordan_wigner(h1_so, h2_so)` es genérica: sirve también para $N$, $S_z$, $S^2$ (#8) y para el tapering (#14). Por defecto exige un operador hermítico (coeficientes reales) y falla si no lo es, porque eso delata un error de índices; para operadores no hermíticos como $S^+ = \sum_p a^\dagger_{p\alpha} a_{p\beta}$ usar `hermitian=False`, que devuelve los coeficientes complejos.

**Signo de los orbitales.** PySCF puede devolver un orbital con el signo invertido entre una distancia y la siguiente. El espectro no cambia, pero los coeficientes de $\hat H$ sí, y eso rompe el *warm start* de #5. Se fija el signo para que el primer coeficiente de cada orbital con $|c| > 10^{-4}$ sea positivo (si algún orbital no tiene ninguno, se lanza un error en vez de anularlo) ([ADR-0003](https://github.com/Santiago-Garrote/QJ/blob/main/docs/adr/0003-signo-orbitales-y-cache.md)). Verificado: ningún término de Pauli cambia de signo de golpe entre distancias vecinas (los únicos cambios, cerca de 0,67 Å, son coeficientes que cruzan cero de forma suave).

**Orbitales RHF en toda la grilla.** RHF es inestable hacia UHF desde ~1,1 Å (`docs/analysis/state-of-the-art.md`; verificado con `mf.stability(external=True)`), pero se mantienen los orbitales RHF: solo definen la base (FCI no cambia), el estado de referencia sigue siendo un singlete y ⟨S²⟩ = 0 sigue sirviendo como diagnóstico en la sección 6 (#8). Con orbitales UHF la referencia quedaría contaminada de espín.

**Caché.** RHF y el Hamiltoniano se calculan una sola vez por distancia (`lru_cache` en funciones privadas). Las funciones públicas devuelven **copias**: `qubit_hamiltonian` devuelve `op.copy()` y `h4_scf` una copia del `mf` con sus propios `mo_coeff`, `mo_energy` y `mo_occ`, así que modificar lo devuelto (o correr `mf.kernel()` / `mf.stability()` sobre él) no altera el caché ni desincroniza orbitales y Hamiltoniano. La clave del caché es `d` redondeada a 6 decimales: `qubit_hamiltonian(0.1 * 7)` reutiliza el de `0.7`.

**Validación** (tabla `ham_check`, las 19 distancias):

| Prueba | Criterio | Resultado |
|---|---|---|
| $\langle \text{HF}\vert op \vert \text{HF}\rangle + e_\text{nuc}$ vs. $E_\text{HF}$ de PySCF | < 1e-8 Ha | máx. ≈ 2e-14 Ha |
| Mínimo de `op` en el sector $N=4$, $S_z=0$ (36 estados) + $e_\text{nuc}$ vs. FCI | < 1e-8 Ha | máx. ≈ 2e-14 Ha |
| Términos de Pauli | — | 185 (184 más la identidad) en todas las distancias, mismo conjunto de Paulis |

`SECTOR` son los 36 índices de la base computacional con 2 α y 2 β, **en el orden `ci[ia, ib]` de PySCF** (`cistring.make_strings`): un vector CI `ci` de PySCF corresponde a `psi[SECTOR] = ci.ravel()` (signos verificados contra `fci.direct_spin1.energy` en estados al azar, error < 3e-14 Ha). Es la misma definición que usa la sección 3 (#4).

La validación contra FCI se hace en el sector $N=4$, $S_z=0$: el mínimo del espectro completo (256 estados) puede tener otro número de electrones. FCI se calcula acá mismo con los orbitales de `h4_scf`; cuando esté `refs` (#1) se puede comparar también contra `refs.e_fci`.

**Qué no funcionó.** La primera versión armaba los productos con `A.compose(B)`, que en Qiskit es la matriz $B \cdot A$, no $A \cdot B$: quedaba $a_Q a_P^\dagger$ en vez de $a_P^\dagger a_Q$, o sea el Hamiltoniano con partículas y huecos intercambiados. Con 4 electrones en 8 espín-orbitales (medio llenado) ese operador tiene **el mismo espectro en el sector** $N=4$, así que la prueba contra FCI pasaba igual; solo la prueba de HF lo detectó (el estado HF aparecía en `11001100`). Se usa `@` (`A @ B` = $A \cdot B$). Por eso las dos pruebas son necesarias.

## 3. Ansatz

## 4. VQE

## 5. Resultados: curva, $R_e$, $D_e$ y tabla (energía, $\sigma$, shots, `cz`)

## 6. Zona estirada ($d \geq 1{,}8$ Å) vs. CCSD

## 7. Decisiones, qué no funcionó y limitaciones